In [1]:
import json
import os
import pandas as pd

In [2]:
results_dir = '/data/rosa/work_in_progress/dictionary_learning_demo/SAEs_shoe_simple_final'
# '/data/rosa/work_in_progress/dictionary_learning_demo/._resid_15__data_rosa_work_in_progress_compositional_interpretability_outputs_shoe_simple_two_level_lr0.0005_epochs30_batch8_warmup100_pythia_cls_head_batch_top_k'

In [3]:
results = {}

for submodule_dir in os.listdir(results_dir):
    for trainer_dir in os.listdir(os.path.join(results_dir, submodule_dir)):
        folder = os.path.join(submodule_dir, trainer_dir)
        run_dir = os.path.join(results_dir, folder)
        try:
            with open(os.path.join(run_dir, 'eval_results.json')) as f:
                metrics = json.load(f)
                config_file = os.path.join(run_dir, 'config.json')
                
                with open(config_file) as f:
                    config = json.load(f)
                    # l1_penalty = config['trainer']['l1_penalty']
                    # k = config['trainer']['k']
                    # target_l0 = config['trainer']['target_l0']
                # results[f'{submodule_dir}_lambda{l1_penalty}'] = metrics
                # results[f'{submodule_dir}_k{k}'] = metrics
                # results[f'{submodule_dir}_target_l0{target_l0}'] = metrics
                
                results[f'{submodule_dir}'] = metrics
        except FileNotFoundError:
            print(f'No eval_results.json in {run_dir}')

In [4]:
# sort by submodule name
results = dict(sorted(results.items()))

In [5]:
results

{'attn_out_0': {'l2_loss': 1.4419174140691757,
  'l1_loss': 10.915248680114747,
  'l0': 88.877265625,
  'frac_variance_explained': 0.8764033442735673,
  'cossim': 0.9749932518601417,
  'l2_ratio': 0.9811988726258278,
  'relative_reconstruction_bias': 1.082961014509201,
  'loss_original': 9.510973978042603,
  'loss_reconstructed': 9.433878939151764,
  'loss_zero': 9.886995177268982,
  'frac_recovered': 0.9067785843841557,
  'frac_alive': 0.10211181640625,
  'hyperparameters': {'n_inputs': 200, 'context_length': 64}},
 'attn_out_1': {'l2_loss': 0.9848997437953949,
  'l1_loss': 35.297924489974974,
  'l0': 83.244921875,
  'frac_variance_explained': 0.918880577981472,
  'cossim': 0.9837337854504585,
  'l2_ratio': 1.001203971505165,
  'relative_reconstruction_bias': 1.0313590061664581,
  'loss_original': 9.510973978042603,
  'loss_reconstructed': 9.479382524490356,
  'loss_zero': 9.729337410926819,
  'frac_recovered': 0.8203007531072944,
  'frac_alive': 0.10174560546875,
  'hyperparameters':

In [6]:
# get a df where the columns are submodule names, frac_variance_explained, l1_loss, l0, frac_alive, and frac_recovered
df = pd.DataFrame(results).T
df = df[['frac_variance_explained', 'l1_loss', 'l0', 'frac_alive', 'frac_recovered', 'loss_original', 'loss_reconstructed']]

# add a column for the difference between the original and reconstructed loss
df['loss_diff'] = df['loss_original'] - df['loss_reconstructed']
df['loss_diff'] = df['loss_diff'].abs()
# drop the original and reconstructed loss columns
df = df.drop(columns=['loss_original', 'loss_reconstructed'])

# sort by submodule names
df = df.sort_index()

# turn fractions into percentages
df['frac_alive'] = 100*df['frac_alive']
df['frac_recovered'] = 100*df['frac_recovered']
df['frac_variance_explained'] = 100*df['frac_variance_explained']
df['l0'] = df['l0'].astype(int)
df['l1_loss'] = df['l1_loss'].astype(int)
df['frac_alive'] = df['frac_alive'].astype(int)
df['frac_recovered'] = df['frac_recovered'].astype(int)
df['frac_variance_explained'] = df['frac_variance_explained'].astype(int)
df = df.apply(pd.to_numeric, errors='coerce')
# keep the first two decimal places for loss_diff
df['loss_diff'] = df['loss_diff'].round(2)
# move the frac_recovery column and its values to the end
df = df[['frac_variance_explained', 'l1_loss', 'l0', 'frac_alive', 'loss_diff', 'frac_recovered']]

# add % sign to the values in the first column
df['frac_variance_explained'] = df['frac_variance_explained'].astype(str) + '%'
df['frac_alive'] = df['frac_alive'].astype(str) + '%'
df['frac_recovered'] = df['frac_recovered'].astype(str) + '%'

# rename the columns
df.columns = ['% Variance Explained', 'L1', 'L0', '% Alive', 'CE Diff', '% CE Recovered']

In [7]:
df

,% Variance Explained,L1,L0,% Alive,CE Diff,% CE Recovered
attn_out_0,87%,10,88,10%,0.08,90%
attn_out_1,91%,35,83,10%,0.03,82%
attn_out_2,94%,77,67,4%,0.00,95%
attn_out_3,98%,54,43,2%,0.02,101%
attn_out_4,91%,20,65,4%,0.01,97%
attn_out_5,89%,19,62,2%,0.00,100%
embed,94%,0,7,1%,0.15,116%
mlp_out_0,97%,5,7,1%,0.04,98%
mlp_out_1,97%,26,62,6%,0.01,93%
mlp_out_2,98%,10,16,5%,0.03,56%
